# STEP 1 — SETUP


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

MERGED_MANIFEST_PATH = Path(
    "/content/drive/MyDrive/Image_captioning/Dataset/Processed/Merged/merged_captions_raw.csv"
)

print("File exists:", MERGED_MANIFEST_PATH.exists())
print("Path:", MERGED_MANIFEST_PATH)

File exists: True
Path: /content/drive/MyDrive/Image_captioning/Dataset/Processed/Merged/merged_captions_raw.csv


In [ ]:
import pandas as pd

captions_df = pd.read_csv(MERGED_MANIFEST_PATH)

print("Shape:", captions_df.shape)
print("Columns:", list(captions_df.columns))
print("Missing captions:", captions_df["caption"].isna().sum())

print("\n✓ Notebook 3 ready")

Shape: (199369, 4)
Columns: ['image_id', 'dataset', 'image_name', 'caption']
Missing captions: 0

✓ Notebook 3 ready


STEP 2: Conservative Caption Cleaning

In [ ]:
import re


def preprocess_caption(caption):
    """
    Conservative preprocessing for image caption generation.

    Keeps natural language intact while removing formatting noise.
    """

    # Convert to string
    caption = str(caption)

    # Convert to lowercase
    caption = caption.lower()

    # Remove leading/trailing spaces
    caption = caption.strip()

    # Normalize multiple spaces
    caption = re.sub(
        r"\s+",
        " ",
        caption
    )

    # Fix spaces before punctuation
    caption = re.sub(
        r"\s+([.,!?;:])",
        r"\1",
        caption
    )

    # Normalize spaces after punctuation
    caption = re.sub(
        r"([.,!?;:])\s*",
        r"\1 ",
        caption
    )

    # Remove unnecessary repeated spaces again
    caption = re.sub(
        r"\s+",
        " ",
        caption
    ).strip()

    return caption


# ------------------------------------------------------------
# APPLY PREPROCESSING
# ------------------------------------------------------------

captions_df["processed_caption"] = (
    captions_df["caption"]
    .apply(preprocess_caption)
)


print("=" * 70)
print("CAPTION PREPROCESSING COMPLETE")
print("=" * 70)

print(
    "\nOriginal caption examples:"
)

for caption in captions_df["caption"].head(5):
    print("-", repr(caption))


print(
    "\nProcessed caption examples:"
)

for caption in captions_df[
    "processed_caption"
].head(5):
    print("-", repr(caption))


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

empty_processed = (
    captions_df[
        "processed_caption"
    ]
    .str.strip()
    .eq("")
    .sum()
)


print(
    "\nTotal captions:",
    len(captions_df)
)

print(
    "Empty processed captions:",
    empty_processed
)


assert (
    empty_processed == 0
)


print(
    "\n✓ Captions cleaned successfully"
)

CAPTION PREPROCESSING COMPLETE

Original caption examples:
- 'A child in a pink dress is climbing up a set of stairs in an entry way .'
- 'A girl going into a wooden building .'
- 'A little girl climbing into a wooden playhouse .'
- 'A little girl climbing the stairs to her playhouse .'
- 'A little girl in a pink dress going into a wooden cabin .'

Processed caption examples:
- 'a child in a pink dress is climbing up a set of stairs in an entry way.'
- 'a girl going into a wooden building.'
- 'a little girl climbing into a wooden playhouse.'
- 'a little girl climbing the stairs to her playhouse.'
- 'a little girl in a pink dress going into a wooden cabin.'

Total captions: 199369
Empty processed captions: 0

✓ Captions cleaned successfully


STEP 3: Add Start and End Tokens

In [ ]:
# ============================================================
# STEP 3 — ADD SEQUENCE TOKENS
# ============================================================

START_TOKEN = "<start>"
END_TOKEN = "<end>"


captions_df["model_caption"] = (
    START_TOKEN
    + " "
    + captions_df["processed_caption"]
    + " "
    + END_TOKEN
)


print("=" * 70)
print("SEQUENCE TOKENS ADDED")
print("=" * 70)


for i in range(5):

    print(
        f"\nCaption {i + 1}:"
    )

    print(
        captions_df.iloc[i][
            "model_caption"
        ]
    )


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

starts_correctly = (
    captions_df[
        "model_caption"
    ]
    .str.startswith(
        START_TOKEN + " "
    )
    .all()
)


ends_correctly = (
    captions_df[
        "model_caption"
    ]
    .str.endswith(
        " " + END_TOKEN
    )
    .all()
)


print(
    "\nStarts correctly:",
    starts_correctly
)

print(
    "Ends correctly:",
    ends_correctly
)


assert starts_correctly
assert ends_correctly


print(
    "\n✓ <start> and <end> tokens added"
)

SEQUENCE TOKENS ADDED

Caption 1:
<start> a child in a pink dress is climbing up a set of stairs in an entry way. <end>

Caption 2:
<start> a girl going into a wooden building. <end>

Caption 3:
<start> a little girl climbing into a wooden playhouse. <end>

Caption 4:
<start> a little girl climbing the stairs to her playhouse. <end>

Caption 5:
<start> a little girl in a pink dress going into a wooden cabin. <end>

Starts correctly: True
Ends correctly: True

✓ <start> and <end> tokens added


STEP 4: Check Caption Lengths

In [ ]:
# ============================================================
# NOTEBOOK 3 — CAPTION LENGTH PERCENTILES
# ============================================================

import numpy as np

print("=" * 70)
print("CAPTION LENGTH PERCENTILES")
print("=" * 70)

for percentile in [50, 75, 90, 95, 99]:

    value = np.percentile(
        captions_df["caption_length"],
        percentile
    )

    print(
        f"{percentile}th percentile: {value:.2f}"
    )

CAPTION LENGTH PERCENTILES
50th percentile: 13.00
75th percentile: 16.00
90th percentile: 20.00
95th percentile: 23.00
99th percentile: 30.00


STEP 5: Analyze Captions Longer Than 32

In [ ]:
# ============================================================
# STEP 5 — CHECK LONG CAPTION OUTLIERS
# ============================================================

MAX_LENGTH = 32

long_caption_mask = (
    captions_df["caption_length"] > MAX_LENGTH
)

long_caption_count = long_caption_mask.sum()

long_caption_percentage = (
    long_caption_count
    / len(captions_df)
    * 100
)

print("=" * 70)
print("LONG CAPTION ANALYSIS")
print("=" * 70)

print("\nMaximum sequence length:", MAX_LENGTH)

print(
    "Captions longer than MAX_LENGTH:",
    long_caption_count
)

print(
    "Percentage affected:",
    f"{long_caption_percentage:.4f}%"
)


print("\nExamples of long captions:\n")

long_caption_examples = (
    captions_df[
        long_caption_mask
    ][
        [
            "caption_length",
            "caption"
        ]
    ]
    .head(10)
)

for _, row in long_caption_examples.iterrows():

    print(
        f"Length: {row['caption_length']}"
    )

    print(
        row["caption"]
    )

    print("-" * 70)

LONG CAPTION ANALYSIS

Maximum sequence length: 32
Captions longer than MAX_LENGTH: 1310
Percentage affected: 0.6571%

Examples of long captions:

Length: 35
A girl in a red and black leotard with white writing is standing next to a man in a cap and a girl in a hot pink and black and white checkered outfit .
----------------------------------------------------------------------
Length: 35
A woman in an orange coat and jeans is squatting on a rock wall while a blonde woman in a red jacket stands next to her on the wall checking her electronic device
----------------------------------------------------------------------
Length: 35
A dark haired woman wearing a brown jacket and fatigue bottoms and a balding man wearing a green sweater and blue jeans with a fishing pole , stand at the foot of the surf .
----------------------------------------------------------------------
Length: 33
A man dressed in black and holding a package in his hands is walking on a sidewalk near a police sign askin

STEP 6: Final Validation

In [ ]:
# ============================================================
# STEP 6 — FINAL CAPTION VALIDATION
# ============================================================

print("=" * 70)
print("FINAL CAPTION VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# CHECK REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "image_id",
    "dataset",
    "image_name",
    "caption",
    "processed_caption",
    "model_caption",
    "caption_length"
]

missing_columns = [
    column
    for column in required_columns
    if column not in captions_df.columns
]

print("\nMissing required columns:")
print(missing_columns)


assert len(missing_columns) == 0


# ------------------------------------------------------------
# CHECK EMPTY VALUES
# ------------------------------------------------------------

print("\nMissing values:")

print(
    captions_df[
        required_columns
    ].isna().sum()
)


assert (
    captions_df[
        "processed_caption"
    ].notna().all()
)

assert (
    captions_df[
        "model_caption"
    ].notna().all()
)


# ------------------------------------------------------------
# CHECK TOKENS
# ------------------------------------------------------------

assert (
    captions_df["model_caption"]
    .str.startswith("<start> ")
    .all()
)

assert (
    captions_df["model_caption"]
    .str.endswith(" <end>")
    .all()
)


# ------------------------------------------------------------
# CHECK LENGTHS
# ------------------------------------------------------------

assert (
    captions_df["caption_length"] >= 3
).all()


print("\nTotal caption records:")
print(len(captions_df))

print("\nUnique images:")
print(
    captions_df["image_id"].nunique()
)

print("\nMaximum observed caption length:")
print(
    captions_df["caption_length"].max()
)

print("\nRecommended training MAX_LENGTH:")
print(MAX_LENGTH)

print("\n✓ ALL CAPTION VALIDATIONS PASSED")

FINAL CAPTION VALIDATION

Missing required columns:
[]

Missing values:
image_id             0
dataset              0
image_name           0
caption              0
processed_caption    0
model_caption        0
caption_length       0
dtype: int64

Total caption records:
199369

Unique images:
39874

Maximum observed caption length:
80

Recommended training MAX_LENGTH:
32

✓ ALL CAPTION VALIDATIONS PASSED


STEP 7: Save Processed Captions

In [ ]:
# ============================================================
# STEP 7 — SAVE PROCESSED CAPTION DATASET
# ============================================================

from pathlib import Path


CAPTION_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Image_captioning/Dataset/Processed/Captions"
)

CAPTION_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PROCESSED_CAPTION_PATH = (
    CAPTION_OUTPUT_DIR
    / "captions_processed.csv"
)


# Keep only the useful columns
processed_captions_df = captions_df[
    [
        "image_id",
        "dataset",
        "image_name",
        "caption",
        "processed_caption",
        "model_caption",
        "caption_length"
    ]
].copy()


processed_captions_df.to_csv(
    PROCESSED_CAPTION_PATH,
    index=False
)


print("=" * 70)
print("PROCESSED CAPTIONS SAVED")
print("=" * 70)

print("\nPath:")
print(PROCESSED_CAPTION_PATH)

print("\nShape:")
print(processed_captions_df.shape)

print(
    "\nFile size:",
    f"{PROCESSED_CAPTION_PATH.stat().st_size / (1024 * 1024):.2f} MB"
)

print("\n✓ Processed caption dataset saved successfully")

PROCESSED CAPTIONS SAVED

Path:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Captions/captions_processed.csv

Shape:
(199369, 7)

File size: 49.24 MB

✓ Processed caption dataset saved successfully


STEP 8: Final Reload Verification

In [ ]:
# ============================================================
# STEP 8 — NOTEBOOK 3 FINAL VERIFICATION
# ============================================================

final_captions_df = pd.read_csv(
    PROCESSED_CAPTION_PATH
)


print("=" * 70)
print("NOTEBOOK 3 — FINAL VERIFICATION")
print("=" * 70)


print("\nShape:")
print(final_captions_df.shape)

print("\nColumns:")
print(
    list(final_captions_df.columns)
)

print("\nTotal captions:")
print(
    len(final_captions_df)
)

print("\nUnique images:")
print(
    final_captions_df["image_id"].nunique()
)

print("\nMissing processed captions:")
print(
    final_captions_df[
        "processed_caption"
    ].isna().sum()
)

print("\nMissing model captions:")
print(
    final_captions_df[
        "model_caption"
    ].isna().sum()
)


assert len(final_captions_df) == 199369

assert (
    final_captions_df[
        "processed_caption"
    ].notna()
    .all()
)

assert (
    final_captions_df[
        "model_caption"
    ].notna()
    .all()
)


print("\nRecommended MAX_LENGTH for training:")
print(32)

print("\n" + "=" * 70)
print("✓ NOTEBOOK 3 COMPLETE")
print("=" * 70)

print("\nOutput:")
print(PROCESSED_CAPTION_PATH)

NOTEBOOK 3 — FINAL VERIFICATION

Shape:
(199369, 7)

Columns:
['image_id', 'dataset', 'image_name', 'caption', 'processed_caption', 'model_caption', 'caption_length']

Total captions:
199369

Unique images:
39874

Missing processed captions:
0

Missing model captions:
0

Recommended MAX_LENGTH for training:
32

✓ NOTEBOOK 3 COMPLETE

Output:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Captions/captions_processed.csv
